In [2]:
pip install requests

    tinycss2 (>=1.1.0<1.2) ; extra == 'css'
             ~~~~~~~~^
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [idna]  WARNING: The script normalizer is installed in '/Library/Frameworks/Python.framework/Versions/3.10/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]4/5 [requests]
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 10.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 12.5 MB/s  0:00:00m0:00:0100:01
    tinycss2 (>=1.1.0<1.2) ; extra == 'css'
             ~~~~~~~~^
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 2/4 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/Library/Frameworks/Python.framework/Versions/3.10/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import time
import re
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

In [6]:
BASE_URL = "https://openthebooks.com/virginia-state-employees/"
OUTPUT_CSV = "virginia_state_employees_salaries.csv"

# A browser-like header helps reduce the chance of being blocked
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def clean_text(value: str) -> str:
    """
    Clean text extracted from HTML cells.

    What it does:
    - converts None to empty string
    - strips leading/trailing whitespace
    - collapses repeated internal whitespace
    """
    if value is None:
        return ""
    value = value.strip()
    value = re.sub(r"\s+", " ", value)
    return value


def parse_currency(value: str):
    """
    Convert a salary string like '$635,397.65' into a float.

    Returns:
    - float for valid currency-like strings
    - None for empty or invalid values
    """
    value = clean_text(value)
    if not value:
        return None

    value = value.replace("$", "").replace(",", "")
    try:
        return float(value)
    except ValueError:
        return None


def get_soup(url: str, session: requests.Session) -> BeautifulSoup:
    """
    Request a page and return a BeautifulSoup object.

    Raises an HTTP error if the request fails.
    """
    response = session.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def extract_table_rows(soup: BeautifulSoup) -> list[dict]:
    """
    Extract salary rows from the main OpenTheBooks salary table.

    Expected columns:
    Year, Employer, Name, Title, Annual Wages, Source

    Returns:
    A list of dictionaries, one dictionary per row.
    """
    table = soup.find("table", class_="employer-detail-table")
    if table is None:
        raise ValueError("Could not find the salary table with class 'employer-detail-table'.")

    rows = table.find_all("tr")
    records = []

    # The first row is usually the header row with <th>.
    # We only want rows that contain <td>.
    for row in rows:
        cells = row.find_all("td")
        if not cells:
            continue

        # We expect 6 columns in the table shown by the user
        if len(cells) < 6:
            continue

        year = clean_text(cells[0].get_text(" ", strip=True))
        employer = clean_text(cells[1].get_text(" ", strip=True))
        name = clean_text(cells[2].get_text(" ", strip=True))
        title = clean_text(cells[3].get_text(" ", strip=True))
        annual_wages_raw = clean_text(cells[4].get_text(" ", strip=True))
        source = clean_text(cells[5].get_text(" ", strip=True))

        record = {
            "year": int(year) if year.isdigit() else None,
            "employer": employer,
            "name": name,
            "title": title,
            "annual_wages_raw": annual_wages_raw,
            "annual_wages": parse_currency(annual_wages_raw),
            "source": source,
        }
        records.append(record)

    return records


def find_next_page(soup: BeautifulSoup, current_url: str) -> str | None:
    """
    Try to locate a 'next page' link.

    This function uses several common patterns because websites
    label pagination differently.

    Returns:
    - absolute URL of the next page
    - None if no next page is found
    """
    # Look through all links and try to identify one that means "next"
    for link in soup.find_all("a", href=True):
        text = clean_text(link.get_text(" ", strip=True)).lower()
        classes = " ".join(link.get("class", [])).lower()
        href = link["href"]

        next_text_matches = {
            "next",
            "next page",
            ">",
            "›",
            "»",
        }

        # Match by visible text
        if text in next_text_matches:
            return urljoin(current_url, href)

        # Match by class names or rel attribute
        rel = " ".join(link.get("rel", [])).lower()
        if "next" in classes or rel == "next":
            return urljoin(current_url, href)

        # Some sites use aria-label
        aria_label = clean_text(link.get("aria-label", "")).lower()
        if "next" in aria_label:
            return urljoin(current_url, href)

    return None


def scrape_all_pages(start_url: str, delay_seconds: float = 1.0, max_pages: int = 500) -> pd.DataFrame:
    """
    Scrape the salary table from the starting page and continue
    through pagination until no next page is found.

    Parameters:
    - start_url: first page to scrape
    - delay_seconds: polite delay between requests
    - max_pages: safety stop to avoid infinite loops

    Returns:
    - pandas DataFrame with all scraped records
    """
    session = requests.Session()
    visited_urls = set()
    all_records = []

    current_url = start_url
    page_count = 0

    while current_url and page_count < max_pages:
        if current_url in visited_urls:
            print(f"Stopping because this page was already visited: {current_url}")
            break

        print(f"Scraping page {page_count + 1}: {current_url}")
        visited_urls.add(current_url)

        soup = get_soup(current_url, session)
        page_records = extract_table_rows(soup)
        print(f"  Found {len(page_records)} records on this page")

        all_records.extend(page_records)

        next_url = find_next_page(soup, current_url)
        current_url = next_url
        page_count += 1

        if current_url:
            time.sleep(delay_seconds)

    df = pd.DataFrame(all_records)

    # Remove exact duplicate rows, which can happen on some paginated scrapes
    if not df.empty:
        df = df.drop_duplicates().reset_index(drop=True)

    return df


def main():
    df = scrape_all_pages(BASE_URL, delay_seconds=1.0)

    if df.empty:
        print("No data was scraped.")
        return

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved {len(df):,} rows to {OUTPUT_CSV}")
    print("\nPreview:")
    print(df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Scraping page 1: https://openthebooks.com/virginia-state-employees/


HTTPError: 403 Client Error: Forbidden for url: https://www.openthebooks.com/virginia-state-employees/